In [15]:
!pip install pydicom scikit-learn tqdm

In [16]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from student_template import SimplePneumoniaClassifier, calibrate_fair_thresholds_from_arrays

In [17]:
DATA_DIR = "data/rsna-pneumonia-detection-challenge"
IMG_DIR = os.path.join(DATA_DIR, "stage_2_train_images")
LABELS_PATH = os.path.join(DATA_DIR, "stage_2_train_labels.csv")

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [18]:
raw_labels = pd.read_csv(LABELS_PATH)

df = (
    raw_labels
    .groupby("patientId", as_index=False)
    .agg(Target=("Target", "max"))
)

df["path"] = df["patientId"].apply(lambda x: os.path.join(IMG_DIR, f"{x}.dcm"))

df = df[df["path"].apply(os.path.exists)].reset_index(drop=True)

print(df.head())
print(df["Target"].value_counts())

                              patientId  Target  \
0  0004cfab-14fd-4e49-80ba-63a80b6bddd6       0   
1  000924cf-0f8d-42bd-9158-1af53881a557       0   
2  000db696-cf54-4385-b10b-6b16fbb3f985       1   
3  000fe35a-2649-43d4-b027-e67796d412e0       1   
4  001031d9-f904-4a23-b3e5-2c088acd19c6       1   

                                                path  
0  data/rsna-pneumonia-detection-challenge\stage_...  
1  data/rsna-pneumonia-detection-challenge\stage_...  
2  data/rsna-pneumonia-detection-challenge\stage_...  
3  data/rsna-pneumonia-detection-challenge\stage_...  
4  data/rsna-pneumonia-detection-challenge\stage_...  
Target
0    20672
1     6012
Name: count, dtype: int64


In [19]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Target"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(len(train_df), len(val_df))

21347 5337


In [20]:
class RSNAPneumoniaDataset(Dataset):
    def __init__(self, dataframe, image_size=224, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        dcm = pydicom.dcmread(row["path"])
        image = dcm.pixel_array.astype(np.float32)

        # Some DICOM images are inverted
        if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
            image = image.max() - image

        # Normalize to [0, 1]
        image = image - image.min()
        image = image / (image.max() + 1e-8)

        # Simple augmentation
        if self.train and np.random.rand() < 0.5:
            image = np.flip(image, axis=1).copy()

        image = torch.from_numpy(image).unsqueeze(0).unsqueeze(0)
        image = F.interpolate(
            image,
            size=(self.image_size, self.image_size),
            mode="bilinear",
            align_corners=False
        )
        image = image.squeeze(0)  # [1, H, W]

        label = torch.tensor(row["Target"], dtype=torch.float32)

        sex = getattr(dcm, "PatientSex", "Unknown")

        return image, label, sex

In [21]:
train_dataset = RSNAPneumoniaDataset(train_df, image_size=128, train=True)
val_dataset = RSNAPneumoniaDataset(val_df, image_size=128, train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [22]:
for images, labels, sex in train_loader:
    print(images.shape)
    print(labels.shape)
    print(sex[:5])
    break

torch.Size([16, 1, 128, 128])
torch.Size([16])
('M', 'M', 'M', 'F', 'M')


In [23]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = SimplePneumoniaClassifier(checkpoint_dir=CHECKPOINT_DIR).to(device)

cpu


In [24]:
num_pos = train_df["Target"].sum()
num_neg = len(train_df) - num_pos

pos_weight = torch.tensor([num_neg / max(num_pos, 1)], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [25]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for images, labels, sex in tqdm(loader, leave=False):
        images = images.to(device)
        labels = labels.to(device).view(-1)

        logits = model.forward_logits(images).view(-1)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    all_probs = []
    all_labels = []
    all_sex = []

    with torch.no_grad():
        for images, labels, sex in tqdm(loader, leave=False):
            images = images.to(device)
            labels = labels.to(device).view(-1)

            logits = model.forward_logits(images).view(-1)
            loss = criterion(logits, labels)

            probs = torch.sigmoid(logits)

            total_loss += loss.item() * images.size(0)

            all_probs.extend(probs.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())
            all_sex.extend(list(sex))

    auc = roc_auc_score(all_labels, all_probs)

    return {
        "loss": total_loss / len(loader.dataset),
        "auc": auc,
        "probs": np.array(all_probs),
        "labels": np.array(all_labels),
        "sex": np.array(all_sex)
    }

In [26]:
best_auc = 0.0
num_epochs = 3

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_auc={val_metrics['auc']:.4f}"
    )

    if val_metrics["auc"] > best_auc:
        best_auc = val_metrics["auc"]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_auc": best_auc,
            },
            os.path.join(CHECKPOINT_DIR, "best_model.pt")
        )

        print("Saved new best model")

Epoch 01 | train_loss=0.8774 | val_loss=0.8536 | val_auc=0.8070
Saved new best model


Epoch 02 | train_loss=0.8346 | val_loss=0.8009 | val_auc=0.8251
Saved new best model


Epoch 03 | train_loss=0.8127 | val_loss=0.7835 | val_auc=0.8325
Saved new best model


In [34]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")

model = SimplePneumoniaClassifier(checkpoint_dir=CHECKPOINT_DIR).to(device)
checkpoint = model.load_checkpoint(checkpoint_path)
model.to(device)

print("Loaded checkpoint AUC:", checkpoint.get("val_auc"))

Loaded checkpoint AUC: 0.8529958131047207


In [35]:
val_metrics = evaluate(model, val_loader, criterion, device)

fair_metrics = calibrate_fair_thresholds_from_arrays(
    model=model,
    probabilities=val_metrics["probs"],
    labels=val_metrics["labels"],
    sex_attribute=val_metrics["sex"]
)

print(fair_metrics)
print("Male threshold:", model.threshold_male.item())
print("Female threshold:", model.threshold_female.item())

{'threshold_male': 0.5249999999999999, 'threshold_female': 0.5149999999999999, 'prediction_rate_male': 0.41070277240490005, 'prediction_rate_female': 0.41252796420581656, 'prediction_rate_gap': 0.0018251918009165036, 'tpr_male': 0.8260869565217391, 'tpr_female': 0.8241308793456033, 'tpr_gap': 0.001956077176135884, 'overall_positive_rate': 0.4114671163575042, 'base_positive_rate': 0.42701892448941353}
Male threshold: 0.5249999761581421
Female threshold: 0.5149999856948853


In [36]:
torch.save(
    {
        "epoch": checkpoint.get("epoch", None),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": checkpoint.get("optimizer_state_dict", None),
        "val_auc": val_metrics["auc"],
        "fair_metrics": fair_metrics,
        "threshold_male": float(model.threshold_male.item()),
        "threshold_female": float(model.threshold_female.item()),
        "threshold_default": float(model.threshold_default.item()),
    },
    os.path.join(CHECKPOINT_DIR, "best_model.pt")
)

print("Final checkpoint saved")

Final checkpoint saved


In [37]:
from student_template import SimplePneumoniaClassifier, get_importance_heatmaps, fair_predict

device = "cuda" if torch.cuda.is_available() else "cpu"

test_model = SimplePneumoniaClassifier()
test_model.load_checkpoint("checkpoints/best_model.pt")
test_model.to(device)
test_model.eval()

sample_image, sample_label, sample_sex = val_dataset[0]

prediction = test_model.predict(sample_image, device=device)
print(prediction)

heatmaps = get_importance_heatmaps(test_model, [sample_image], window_size=32, stride=16)
print(type(heatmaps), heatmaps[0].shape, heatmaps[0].min(), heatmaps[0].max())

fair_result = fair_predict(test_model, [sample_image], [sample_sex])
print(fair_result)

{'probability': 0.6954007148742676, 'class': 1, 'label': 'Pneumonia'}
<class 'list'> (128, 128) 0.0 1.0
[{'probability': 0.6954007148742676, 'threshold': 0.5149999856948853, 'class': 1, 'label': 'Pneumonia'}]


In [31]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_model.pt")

model = SimplePneumoniaClassifier(checkpoint_dir=CHECKPOINT_DIR).to(device)
checkpoint = model.load_checkpoint(checkpoint_path)
model.to(device)

best_auc = checkpoint.get("val_auc", 0.0)

print("Continue from AUC:", best_auc)

Continue from AUC: 0.8324660430922264


In [32]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-5,
    weight_decay=1e-4
)

In [33]:
for epoch in range(4, 8):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_metrics = evaluate(model, val_loader, criterion, device)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} | "
        f"val_auc={val_metrics['auc']:.4f}"
    )

    if val_metrics["auc"] > best_auc:
        best_auc = val_metrics["auc"]

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_auc": best_auc,
            },
            os.path.join(CHECKPOINT_DIR, "best_model.pt")
        )

        print("Saved new best model")

Epoch 04 | train_loss=0.7867 | val_loss=0.7676 | val_auc=0.8387
Saved new best model


Epoch 05 | train_loss=0.7740 | val_loss=0.7640 | val_auc=0.8436
Saved new best model


Epoch 06 | train_loss=0.7666 | val_loss=0.7546 | val_auc=0.8447
Saved new best model


Epoch 07 | train_loss=0.7591 | val_loss=0.7467 | val_auc=0.8530
Saved new best model


In [38]:
import os
import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm

from student_template import get_importance_heatmaps

# raw_labels должен уже быть загружен раньше:
# raw_labels = pd.read_csv(LABELS_PATH)

def get_patient_boxes(patient_id):
    rows = raw_labels[
        (raw_labels["patientId"] == patient_id) &
        (raw_labels["Target"] == 1)
    ]

    boxes = []

    for _, row in rows.iterrows():
        if pd.isna(row["x"]):
            continue

        boxes.append((
            float(row["x"]),
            float(row["y"]),
            float(row["width"]),
            float(row["height"])
        ))

    return boxes


def heatmap_bbox_coverage(heatmap, boxes, original_h, original_w):
    """
    heatmap: [H, W] after resize, e.g. 128x128
    boxes: original bbox coordinates from CSV
    original_h/original_w: original DICOM size
    """
    h, w = heatmap.shape

    mask = np.zeros((h, w), dtype=bool)

    scale_x = w / original_w
    scale_y = h / original_h

    for x, y, bw, bh in boxes:
        x1 = int(round(x * scale_x))
        y1 = int(round(y * scale_y))
        x2 = int(round((x + bw) * scale_x))
        y2 = int(round((y + bh) * scale_y))

        x1 = max(0, min(w, x1))
        x2 = max(0, min(w, x2))
        y1 = max(0, min(h, y1))
        y2 = max(0, min(h, y2))

        mask[y1:y2, x1:x2] = True

    total_importance = heatmap.sum()

    if total_importance <= 1e-8:
        return 0.0

    inside_importance = heatmap[mask].sum()

    return float(inside_importance / total_importance)


positive_val_patients = val_df[val_df["Target"] == 1]["patientId"].tolist()

coverages = []

# Можно взять 100, чтобы не ждать слишком долго
for patient_id in tqdm(positive_val_patients[:100]):
    boxes = get_patient_boxes(patient_id)

    if len(boxes) == 0:
        continue

    dcm_path = os.path.join(IMG_DIR, f"{patient_id}.dcm")
    dcm = pydicom.dcmread(dcm_path)

    original_h, original_w = dcm.pixel_array.shape

    # Берём image из val_dataset по patientId
    idx = val_df.index[val_df["patientId"] == patient_id][0]
    image, label, sex = val_dataset[idx]

    heatmap = get_importance_heatmaps(
        test_model,
        [image],
        window_size=32,
        stride=16
    )[0]

    coverage = heatmap_bbox_coverage(
        heatmap,
        boxes,
        original_h=original_h,
        original_w=original_w
    )

    coverages.append(coverage)

print("N:", len(coverages))
print("Mean coverage:", np.mean(coverages))
print("Median coverage:", np.median(coverages))
print("Threshold for full points:", 0.35)

100%|██████████| 100/100 [00:37<00:00,  2.63it/s]

N: 100
Mean coverage: 0.24665862077381462
Median coverage: 0.2185879945755005
Threshold for full points: 0.35
